# House Prices — V7: Sale-Condition Blend

**Recorded Kaggle log RMSE** 0.11194

This notebook recreates the scored submission from saved model predictions and the repository data. Run the cells in order from a Python kernel with the packages in [final_versions/requirements.txt](final_versions/requirements.txt) installed.

## 1. Load the competition data

In [1]:
from io import BytesIO
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

current = Path.cwd()
root = next(
    (path for path in (current, *current.parents)
     if (path / "final_versions" / "rebuild.py").is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Open this notebook from the project repository.")
sys.path.insert(0, str(root))
from final_versions import rebuild

data = root / "Dataset"
if not data.is_dir():
    data = root / "house-prices-advanced-regression-techniques"
test = pd.read_csv(data / "test.csv", keep_default_na=False, na_values=["", "NA"])
assert len(test) == 1459 and test["Id"].is_unique

version = "V7"

## 2. Build the predictions

For normal sales, V7 combines V6, TabICLv2, medium Kumo and large
Kumo with effective log-price weights of 18.75%, 6.25%, 37.5%
and 37.5%. The saved Kumo predictions are the mean of the medium
and large models. Other sales use V6 prices.

In [2]:
v6 = (rebuild.SUBMISSIONS / "v6.csv").read_bytes()
payload = rebuild.rebuild_v7(test, v6)
normal = test["SaleCondition"].eq("Normal")
np.testing.assert_allclose(
    rebuild.read_prices(payload)[~normal],
    rebuild.read_prices(v6)[~normal],
    rtol=1e-12, atol=1e-7,
)
display(pd.DataFrame({
    "Sale condition": ["Normal", "Other"],
    "Houses": [normal.sum(), (~normal).sum()],
    "Prediction": ["V6, TabICLv2 and Kumo blend", "V6"],
}))

,Sale condition,Houses,Prediction
0,Normal,1204,"V6, TabICLv2 and Kumo blend"
1,Other,255,V6


## 3. Verify and save the submission

In [3]:
rebuild.compare_with_submission(version, payload, test["Id"])
output = root / "rebuilt" / rebuild.VERSION_FILES[version]
output.parent.mkdir(parents=True, exist_ok=True)
output.write_bytes(payload)
submission = pd.read_csv(output)
scored = rebuild.SUBMISSIONS / output.name
display(pd.DataFrame([{
    "Output": str(output.relative_to(root)),
    "Rows": len(submission),
    "Comparison with scored CSV": (
        "Byte-identical" if payload == scored.read_bytes() else "Numerically matched"
    ),
}]))
display(submission.head(5).round(2))

,Output,Rows,Comparison with scored CSV
0,rebuilt/v7.csv,1459,Byte-identical


,Id,SalePrice
0,1461,122413.87
1,1462,161599.65
2,1463,187559.05
3,1464,198818.62
4,1465,182330.24


## Results

The blend applied to **1,204 normal sales**. The remaining **255**
retained their V6 prices. The recorded Kaggle log RMSE was **0.11194**.